# Walkthrough: exchange-to-ledger reconciliation

This notebook runs the reconciliation step by step on the synthetic sample data, so each stage
can be inspected. It uses the same functions as the command-line tool (`recon.py`). Nothing new
is calculated here.

**Steps**
1. Load the config and look at the standing rules
2. Load the two input files
3. See why the timezone conversion matters
4. Run the reconciliation
5. Look at each category: TIMING, STANDING, NEEDS_REVIEW
6. Read a write-up the way the morning reviewer would
7. Check the results against the answer key of planted breaks

All data is synthetic. Run `python generate_sample_data.py` first if the `data/` folder is empty.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# The notebook lives in notebooks/; the project folder is one level up.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import recon

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## 1. Config and standing rules

Everything a reviewer might tune is in `config.yaml`: the timezone, the tolerances, and the
standing rules. Each standing rule has an ID, an owner and a description, so every STANDING
break can be traced back to a documented, owned exception.

In [2]:
config = recon.load_config(ROOT / "config.yaml")
print("Timezone:  ", config["reporting_timezone"])
print("Tolerances:", config["tolerances"])

pd.DataFrame(config["standing_rules"])[["id", "name", "type", "owner", "description"]]

Timezone:   America/Puerto_Rico
Tolerances: {'quantity': 1e-08, 'date_window_days': 2}


,id,name,type,owner,description
0,STND-001,Dust,unmatched_below_usd,Crypto Ops - Fund Accounting,Exchange-side movements worth less than $1.00 ...
1,STND-002,Staking rewards,unmatched_txn_type,Crypto Ops - Staking,The exchange credits rewards daily; the ledger...
2,STND-003,Fee rounding,paired_quantity_rounding,Crypto Ops - Fund Accounting,The exchange reports fees to 8 decimals; the l...


## 2. The two input files

The exchange export has one row per transaction, timestamped in UTC. The ledger has one row per
entry, with `external_ref` pointing back to the exchange's transaction ID. That reference is the
matching key.

In [3]:
exchange = recon.load_exchange(ROOT / "data" / "exchange.csv", config["reporting_timezone"])
ledger = recon.load_ledger(ROOT / "data" / "ledger.csv")

print(f"Exchange rows: {len(exchange)}   Ledger rows: {len(ledger)}")
exchange.head()

Exchange rows: 200   Ledger rows: 182


,exchange_txn_id,timestamp_utc,txn_type,asset,quantity,usd_value,local_date
0,EX-107540,2026-10-01T06:45:40Z,DEPOSIT,BTC,2.983400,186002.40,2026-10-01
1,EX-315173,2026-10-01T06:53:48Z,STAKING_REWARD,SOL,0.113609,16.89,2026-10-01
2,EX-801978,2026-10-01T07:36:13Z,DEPOSIT,BTC,0.718500,44795.44,2026-10-01
3,EX-739717,2026-10-01T11:41:04Z,DEPOSIT,USDC,82673.560000,82673.56,2026-10-01
4,EX-401338,2026-10-01T11:47:05Z,DEPOSIT,BTC,3.192700,199051.37,2026-10-01


In [4]:
ledger.head()

,ledger_entry_id,external_ref,booking_date,txn_type,asset,quantity,usd_value,memo
0,LG-000001,EX-107540,2026-10-01,DEPOSIT,BTC,2.9834,186002.40,Deposit BTC
1,LG-000002,EX-349565,2026-10-01,TRADE_SELL,BTC,-0.1882,-11733.48,Trade Sell BTC
2,LG-000003,EX-350921,2026-10-01,FEE,USDC,-91.7200,-91.72,Fee USDC
3,LG-000004,EX-401338,2026-10-01,DEPOSIT,BTC,3.1927,199051.37,Deposit BTC
4,LG-000005,EX-428914,2026-10-01,TRADE_SELL,ETH,-21.9921,-51778.20,Trade Sell ETH


## 3. Why the timezone conversion matters

Exchanges report in UTC. San Juan is UTC-4, so anything after 8 PM local time is already the
next day in UTC. The ledger books in local dates. Comparing UTC dates would create false timing
breaks on every late-evening trade.

`load_exchange` adds a `local_date` column. These are the rows where the two dates differ:

In [5]:
utc_date = pd.to_datetime(exchange["timestamp_utc"]).dt.date
crosses_midnight = exchange[utc_date != exchange["local_date"]]

print(f"{len(crosses_midnight)} exchange rows have a different date in UTC than in San Juan.")
crosses_midnight[["exchange_txn_id", "timestamp_utc", "local_date", "txn_type", "asset", "quantity"]].head()

37 exchange rows have a different date in UTC than in San Juan.


,exchange_txn_id,timestamp_utc,local_date,txn_type,asset,quantity
8,EX-885395,2026-10-02T01:16:37Z,2026-10-01,TRADE_SELL,BTC,-0.7687
9,EX-350921,2026-10-02T02:10:19Z,2026-10-01,FEE,USDC,-91.7200
20,EX-905934,2026-10-03T00:21:07Z,2026-10-02,WITHDRAWAL,USDC,-187968.5700
32,EX-412942,2026-10-04T00:32:38Z,2026-10-03,DEPOSIT,BTC,4.3655
46,EX-786516,2026-10-05T00:56:49Z,2026-10-04,DEPOSIT,USDC,50000.0000


## 4. Run the reconciliation

`reconcile` does the matching and classification and returns:
- `breaks`: one line per break, with category, reason code and rule ID
- `stats`: how many references were compared and how many were clean

Clean pairs (everything agrees) are counted but not listed.

In [6]:
breaks, stats = recon.reconcile(exchange, ledger, config)
stats

{'exchange_rows': 200,
 'ledger_rows': 182,
 'references_compared': 200,
 'clean_pairs': 166}

In [7]:
# How many break lines per category and reason code
pd.crosstab(breaks["reason_code"], breaks["category"], margins=True, margins_name="Total")

category,NEEDS_REVIEW,STANDING,TIMING,Total
reason_code,,,,
AMBIGUOUS_STANDING_RULE,1,0,0,1
ASSET_MISMATCH,1,0,0,1
DATE_OUTSIDE_WINDOW,1,0,0,1
DATE_WITHIN_WINDOW,0,0,3,3
DUPLICATE_EXCHANGE_ID,2,0,0,2
DUPLICATE_LEDGER_REF,2,0,0,2
MISSING_IN_LEDGER,2,0,0,2
MISSING_ON_EXCHANGE,1,0,0,1
MULTIPLE_DIFFERENCES,1,0,0,1


## 5. Each category

### TIMING
The same transaction, the same asset and quantity, booked up to 2 days apart. These should
clear on their own.

In [8]:
columns = ["break_id", "txn_ref", "txn_type", "exchange_asset", "exchange_quantity", "exchange_date", "ledger_date", "day_gap"]
breaks[breaks["category"] == "TIMING"][columns]

,break_id,txn_ref,txn_type,exchange_asset,exchange_quantity,exchange_date,ledger_date,day_gap
14,BRK-015,EX-196126,TRADE_SELL,SOL,-340.00,2026-10-13,2026-10-14,1.0
15,BRK-016,EX-389282,DEPOSIT,BTC,1.25,2026-10-03,2026-10-04,1.0
16,BRK-017,EX-989227,WITHDRAWAL,ETH,-12.50,2026-10-09,2026-10-11,2.0


### STANDING
Known, documented differences. Each line carries the rule that matched it and the rule's owner.

In [9]:
standing = breaks[breaks["category"] == "STANDING"]
standing.groupby(["rule_id", "rule_owner"]).agg(
    lines=("break_id", "count"),
    exchange_usd=("exchange_usd", "sum"),
).round(2)

,,lines,exchange_usd
rule_id,rule_owner,,
STND-001,Crypto Ops - Fund Accounting,2,0.87
STND-002,Crypto Ops - Staking,14,220.80
STND-003,Crypto Ops - Fund Accounting,3,-57.35


In [10]:
# The fee-rounding items: the quantity differs only in the 7th-8th decimal place.
# Shown with 8 decimals; the default display rounds to 6 and would hide the difference.
fees = standing[standing["rule_id"] == "STND-003"][["break_id", "txn_ref", "exchange_quantity", "ledger_quantity", "quantity_diff"]]
fees.style.format("{:.8f}", subset=["exchange_quantity", "ledger_quantity", "quantity_diff"])

,break_id,txn_ref,exchange_quantity,ledger_quantity,quantity_diff
23,BRK-024,EX-232321,-0.00031297,-0.00031300,0.00000003
27,BRK-028,EX-437758,-0.00048761,-0.00048800,0.00000039
29,BRK-030,EX-565108,-0.00012345,-0.00012300,-0.00000045


### NEEDS_REVIEW
Everything that needs a person. These are never auto-cleared. The `detail` column says what the
tool saw.

In [11]:
review = breaks[breaks["category"] == "NEEDS_REVIEW"]
review[["break_id", "reason_code", "rule_id", "txn_ref", "presence", "exchange_asset", "ledger_asset",
        "exchange_quantity", "ledger_quantity", "detail"]]

,break_id,reason_code,rule_id,txn_ref,presence,exchange_asset,ledger_asset,exchange_quantity,ledger_quantity,detail
0,BRK-001,AMBIGUOUS_STANDING_RULE,CORE-AMBIGUOUS,EX-865054,exchange_only,ETH,NaN,0.000150,NaN,"Matches more than one standing rule: STND-001,..."
1,BRK-002,ASSET_MISMATCH,CORE-ASSET,EX-786516,both,USDC,USDT,50000.000000,50000.000000,Asset differs between exchange and ledger
2,BRK-003,DATE_OUTSIDE_WINDOW,CORE-DATE,EX-533914,both,BTC,BTC,-0.750000,-0.750000,"Dates are 6 days apart, beyond the 2-day window"
3,BRK-004,DUPLICATE_EXCHANGE_ID,CORE-DUPLICATE,EX-550961,both,BTC,BTC,0.400000,0.400000,2 exchange rows share this ID
4,BRK-005,DUPLICATE_EXCHANGE_ID,CORE-DUPLICATE,EX-550961,both,BTC,BTC,0.400000,0.400000,2 exchange rows share this ID
5,BRK-006,DUPLICATE_LEDGER_REF,CORE-DUPLICATE,EX-678116,both,ETH,ETH,-8.000000,-8.000000,"2 ledger entries reference this ID: LG-000104,..."
6,BRK-007,DUPLICATE_LEDGER_REF,CORE-DUPLICATE,EX-678116,both,ETH,ETH,-8.000000,-8.000000,"2 ledger entries reference this ID: LG-000104,..."
7,BRK-008,MISSING_IN_LEDGER,CORE-MISSING,EX-499407,exchange_only,ETH,NaN,3.500000,NaN,"On the exchange, no ledger entry references it"
8,BRK-009,MISSING_IN_LEDGER,CORE-MISSING,EX-713799,exchange_only,USDC,NaN,-25000.000000,NaN,"On the exchange, no ledger entry references it"
9,BRK-010,MISSING_ON_EXCHANGE,CORE-MISSING,EX-575755,ledger_only,NaN,BTC,NaN,0.800000,"In the ledger, but the exchange export has no ..."


## 6. A write-up, as the reviewer sees it

Each NEEDS_REVIEW item gets three sentences in `summary.md`: what is different, the likely
cause, and the decision needed. They come from fixed templates, so the same break always reads
the same way. This is the write-up for the sign-flipped USDC deposit:

In [12]:
sign_flip = review[(review["reason_code"] == "QTY_MISMATCH")
                   & (review["exchange_quantity"] == -review["ledger_quantity"])].iloc[0]

what, cause, decision = recon.write_up(sign_flip.to_dict(), review[review["txn_ref"] == sign_flip["txn_ref"]], config)
print(what)
print(cause)
print(decision)

EX-931621 (deposit, USDC) is 10,000 on the exchange but -10,000 in the ledger, a difference of 20,000.
Likely cause: the entry was booked in the wrong direction (sign flipped).
Decision needed: confirm the correct quantity from the exchange record and decide whether to correct the ledger entry.


## 7. Check against the answer key

`generate_sample_data.py` recorded every break it planted and the bucket it should land in.
Here every planted break is compared with what the tool decided. The automated test
`test_sample_data_matches_answer_key` does the same check.

In [13]:
answer_key = pd.read_csv(ROOT / "data" / "planted_breaks.csv")
actual = breaks[["txn_ref", "category", "reason_code", "rule_id"]].drop_duplicates()

compared = answer_key.merge(actual, on="txn_ref", how="outer", indicator=True)
compared["correct"] = (
    (compared["expected_category"] == compared["category"])
    & (compared["expected_reason_code"] == compared["reason_code"])
    & (compared["expected_rule_id"] == compared["rule_id"])
)

print(f"Planted breaks:           {len(answer_key)}")
print(f"Classified as expected:   {compared['correct'].sum()}")
print(f"Missed by the tool:       {(compared['_merge'] == 'left_only').sum()}")
print(f"Flagged but not planted:  {(compared['_merge'] == 'right_only').sum()}")

Planted breaks:           34
Classified as expected:   34
Missed by the tool:       0
Flagged but not planted:  0


## To produce the daily files

The notebook is for exploring. The daily run uses the command line, which writes
`output/breaks.csv` and `output/summary.md`:

```
python recon.py --exchange data/exchange.csv --ledger data/ledger.csv --config config.yaml
```

See `RUNBOOK.md` for the full daily procedure.